<a href="https://colab.research.google.com/github/ibrah1mr7a/house-price-regression/blob/main/models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

In [16]:
#1.Load data

df = pd.read_csv('/train.csv')
print(df.shape)
print(df.head())

(1460, 81)
   Id  MSSubClass MSZoning  ...  SaleType  SaleCondition SalePrice
0   1          60       RL  ...        WD         Normal    208500
1   2          20       RL  ...        WD         Normal    181500
2   3          60       RL  ...        WD         Normal    223500
3   4          70       RL  ...        WD        Abnorml    140000
4   5          60       RL  ...        WD         Normal    250000

[5 rows x 81 columns]


In [21]:
#2.Explore

print(df.describe())
print(df.isnull().sum().sort_values(ascending=False).head(20))

                Id   MSSubClass  ...       YrSold      SalePrice
count  1460.000000  1460.000000  ...  1460.000000    1460.000000
mean    730.500000    56.897260  ...  2007.815753  180921.195890
std     421.610009    42.300571  ...     1.328095   79442.502883
min       1.000000    20.000000  ...  2006.000000   34900.000000
25%     365.750000    20.000000  ...  2007.000000  129975.000000
50%     730.500000    50.000000  ...  2008.000000  163000.000000
75%    1095.250000    70.000000  ...  2009.000000  214000.000000
max    1460.000000   190.000000  ...  2010.000000  755000.000000

[8 rows x 38 columns]
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical    

In [23]:
#Corellation with target

numeric_df = df.select_dtypes(include=[np.number])
print(numeric_df.corr()["SalePrice"].sort_values(ascending=False).head(10))

SalePrice        1.000000
OverallQual      0.790982
GrLivArea        0.708624
GarageCars       0.640409
GarageArea       0.623431
TotalBsmtSF      0.613581
1stFlrSF         0.605852
FullBath         0.560664
TotRmsAbvGrd     0.533723
YearBuilt        0.522897
YearRemodAdd     0.507101
GarageYrBlt      0.486362
MasVnrArea       0.477493
Fireplaces       0.466929
BsmtFinSF1       0.386420
LotFrontage      0.351799
WoodDeckSF       0.324413
2ndFlrSF         0.319334
OpenPorchSF      0.315856
HalfBath         0.284108
LotArea          0.263843
BsmtFullBath     0.227122
BsmtUnfSF        0.214479
BedroomAbvGr     0.168213
ScreenPorch      0.111447
PoolArea         0.092404
MoSold           0.046432
3SsnPorch        0.044584
BsmtFinSF2      -0.011378
BsmtHalfBath    -0.016844
MiscVal         -0.021190
Id              -0.021917
LowQualFinSF    -0.025606
YrSold          -0.028923
OverallCond     -0.077856
MSSubClass      -0.084284
EnclosedPorch   -0.128578
KitchenAbvGr    -0.135907
Name: SalePr

In [24]:
#3.Split target

X = df.drop(columns=['SalePrice', 'Id'])
y = df['SalePrice']

In [26]:
#separate numeric and catogorical coulmns

numeric_features = X.select_dtypes(include=[np.number]).columns
categorical_features = X.select_dtypes(include=['object']).columns

In [38]:
#4.Preprocessing pipeline

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [39]:
#5.Train / Test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [46]:
#6.Model 1: LinearRegression

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred = lr_pipeline.predict(X_test)

In [47]:
print("\n--- Linear Regression ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test,y_pred)))
print("R2: ", r2_score(y_test, y_pred))


--- Linear Regression ---
RMSE: 65387.6252350318
R2:  0.442586740330448


In [48]:
#7.Model 2: Ridge
Rid_pipeline = Pipeline(steps =[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=10))
])

Rid_pipeline.fit(X_train, y_train)
y_pred_rid = Rid_pipeline.predict(X_test)

In [49]:
print("\n--- Ridge ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test,y_pred_rid)))
print("R2: ", r2_score(y_test, y_pred_rid))


--- Ridge ---
RMSE: 30524.611902659875
R2:  0.8785251230990737


In [52]:
#8.Model 3: Random Forest

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

In [51]:
print("\n--- Random Forest ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test,y_pred_rf)))
print("R2: ", r2_score(y_test, y_pred_rf))


--- Random Forest ---
RMSE: 28863.515219692352
R2:  0.891386299834707


In [69]:
param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(estimator=rf_pipeline, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)


print("Best Parameters:", grid_search.best_params_)
print("Best CV Score (Negative MSE):".format(grid_search.best_score_))
print("Best CV Score (RMSE):".format(np.sqrt(-grid_search.best_score_)))


best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)


print("\n--- Optimized Random Forest ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
print("R2: ", r2_score(y_test, predictions))

Best Parameters: {'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Best CV Score (Negative MSE):
Best CV Score (RMSE):

--- Optimized Random Forest ---
RMSE: 29633.581945710128
R2:  0.8855134507695008


In [63]:
#9.Model 4: XGBoost

Xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(n_estimators=200, random_state=42))
])

Xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = Xgb_pipeline.predict(X_test)


In [64]:
print("\n--- XGBoost ---")
print("RMSE:", np.sqrt(mean_squared_error(y_test,y_pred_xgb)))
print("R2: ", r2_score(y_test, y_pred_xgb))


--- XGBoost ---
RMSE: 27860.73136154182
R2:  0.8988021612167358


In [54]:
#10 Cross_Validation

cv_scores = cross_val_score(
    rf_pipeline, X, y, cv=5, scoring='neg_mean_squared_error'
)
print("Cross-Validation Scores:", -cv_scores.mean())

Cross-Validation Scores: 919981449.4417388


In [60]:
#11 Feature Importance

ohe_coulmns = rf_pipeline.named_steps['preprocessor'] \
.named_transformers_['cat'] \
.get_feature_names_out(categorical_features)

all_feature_names = np.concatenate([numeric_features, ohe_coulmns])

importance = rf_pipeline.named_steps['model'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importance
}).sort_values("Importance", ascending=False)

feature_importance_df.head(15)


,Feature,Importance
3,OverallQual,0.556489
15,GrLivArea,0.122593
11,TotalBsmtSF,0.033923
13,2ndFlrSF,0.029906
12,1stFlrSF,0.026981
8,BsmtFinSF1,0.026582
2,LotArea,0.018132
26,GarageArea,0.015455
25,GarageCars,0.014169
5,YearBuilt,0.012914
